**BRFSS Risk Factors for Heart Disease Data EDA**

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/brfss/brfss_survey_data_2024.csv")

# print first entries
df.head()

# number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 457670
Number of columns: 301


In [7]:
# print all column names
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

# print names of columns and stats on each column 
eda_table = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(dropna=False).values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2)
})

eda_table = eda_table.sort_values("missing_percent", ascending=False)
eda_table.to_csv("eda_basics_table.csv", index=False)

1. _STATE
2. FMONTH
3. IDATE
4. IMONTH
5. IDAY
6. IYEAR
7. DISPCODE
8. SEQNO
9. _PSU
10. CTELENM1
11. PVTRESD1
12. COLGHOUS
13. STATERE1
14. CELPHON1
15. LADULT1
16. NUMADULT
17. RESPSLC1
18. LANDSEX3
19. SAFETIME
20. CTELNUM1
21. CELLFON5
22. CADULT1
23. CELLSEX3
24. PVTRESD3
25. CCLGHOUS
26. CSTATE1
27. LANDLINE
28. HHADULT
29. SEXVAR
30. GENHLTH
31. PHYSHLTH
32. MENTHLTH
33. POORHLTH
34. PRIMINS2
35. PERSDOC3
36. MEDCOST1
37. CHECKUP1
38. EXERANY2
39. LASTDEN4
40. RMVTETH4
41. CVDINFR4
42. CVDCRHD4
43. CVDSTRK3
44. ASTHMA3
45. ASTHNOW
46. CHCSCNC1
47. CHCOCNC1
48. CHCCOPD3
49. ADDEPEV3
50. CHCKDNY2
51. HAVARTH4
52. DIABETE4
53. DIABAGE4
54. MARITAL
55. EDUCA
56. RENTHOM1
57. NUMHHOL4
58. NUMPHON4
59. CPDEMO1C
60. VETERAN3
61. EMPLOY1
62. CHILDREN
63. INCOME3
64. PREGNANT
65. WEIGHT2
66. HEIGHT3
67. DEAF
68. BLIND
69. DECIDE
70. DIFFWALK
71. DIFFDRES
72. DIFFALON
73. HADMAM
74. HOWLONG
75. CERVSCRN
76. CRVCLCNC
77. CRVCLPAP
78. CRVCLHPV
79. HADHYST2
80. HADSIGM4
81. COLNSIGM
82. COLN

In [8]:
eda_numerical_summary = df.describe()
eda_numerical_summary.to_csv("eda_numerical_summary.csv", index=False)

Useful columns:   
**_MICHD** is "Respondents that have ever reported having coronary heart disease (CHD) or myocardial infarction (MI)": 1=Yes, 2=No    
**CVDINFR4** is "(Ever told) you had a heart attack, also called a myocardial infarction?"  
**CVDCRHD4** is "(Ever told) (you had) angina or coronary heart disease?"  


In [ ]:
# function that for each feature calulates the percentage of people in each category that have heart disease
def heart_disease_rate_by_feature(data, feature, target="_MICHD"):
    if feature not in data.columns:
        raise ValueError(f"{feature} is not in the dataframe columns.")
    
    if target not in data.columns:
        raise ValueError(f"{target} is not in the dataframe columns.")

    temp = data[[feature, target]].copy()
    temp = temp[temp[target].isin([1, 2])]
    temp = temp.dropna(subset=[feature])

    summary = (
        temp
        .groupby(feature)
        .agg(
            total_people=(target, "count"),
            heart_disease_cases=(target, lambda x: (x == 1).sum())
        )
        .reset_index()
    )

    summary["heart_disease_percent"] = (
        summary["heart_disease_cases"] / summary["total_people"] * 100
    ).round(2)

    summary = summary.rename(columns={feature: "feature_value"})

    summary.insert(0, "feature", feature)

    summary = summary.sort_values(
        "heart_disease_percent",
        ascending=False
    )

    return summary

In [ ]:
heart_disease_rate_by_feature(df, "_BMI5CAT")

,feature,feature_value,total_people,heart_disease_cases,heart_disease_percent
3,_BMI5CAT,4.0,138046,15015,10.88
2,_BMI5CAT,3.0,145112,14309,9.86
0,_BMI5CAT,1.0,7259,666,9.17
1,_BMI5CAT,2.0,119864,9440,7.88
